#### CREDIT PORTFOLIO HEALTH MONITOR
- MySQL Database Setup & Data Loading

#### Import

In [2]:
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

print("Libraries imported and .env file loaded")

Libraries imported and .env file loaded


#### Load Credentials

In [3]:
DB_HOST = os.getenv('MYSQL_HOST')
DB_PORT = os.getenv('MYSQL_PORT')
DB_NAME = os.getenv('MYSQL_DATABASE')
DB_USER = os.getenv('MYSQL_USER')
DB_PASSWORD = os.getenv('MYSQL_PASSWORD')

# Validation
if not all([DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD]):
    raise ValueError("Missing database credentials in .env file. Please check your .env file.")

print("Database credentials loaded successfully from .env")
print(f"Database: {DB_NAME} | Host: {DB_HOST}")

Database credentials loaded successfully from .env
Database: mophones_credit | Host: localhost


#### Create Database Engine

In [4]:
connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(
    connection_string, 
    pool_pre_ping=True,      
    echo=False              
)

print("SQLAlchemy Engine created successfully")

SQLAlchemy Engine created successfully


#### Create Database

In [5]:
with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME};"))
    conn.execute(text(f"USE {DB_NAME};"))

print(f"Database '{DB_NAME}' is ready!")

Database 'mophones_credit' is ready!


#### Create Professional Schema

In [7]:
schema_statements = [
    "DROP TABLE IF EXISTS repayments;",
    "DROP TABLE IF EXISTS loans;",
    "DROP TABLE IF EXISTS customers;",
    
    """CREATE TABLE customers (
        customer_id      VARCHAR(20) PRIMARY KEY,
        full_name        VARCHAR(100) NOT NULL,
        phone_number     VARCHAR(30),
        email            VARCHAR(100),
        age              INT,
        gender           VARCHAR(10),
        location         VARCHAR(50),
        income_band      VARCHAR(20),
        credit_score     INT,
        created_at       DATE,
        INDEX idx_location (location),
        INDEX idx_income_band (income_band),
        INDEX idx_credit_score (credit_score)
    ) ENGINE=InnoDB;""",
    
    """CREATE TABLE loans (
        loan_id          VARCHAR(20) PRIMARY KEY,
        customer_id      VARCHAR(20) NOT NULL,
        disbursed_date   DATE NOT NULL,
        principal        DECIMAL(12,2) NOT NULL,
        term_months      INT NOT NULL,
        interest_rate    DECIMAL(5,2),
        device_model     VARCHAR(60),
        promo_applied    BOOLEAN DEFAULT FALSE,
        
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ON DELETE RESTRICT,
        INDEX idx_disbursed (disbursed_date),
        INDEX idx_device_model (device_model),
        INDEX idx_promo (promo_applied)
    ) ENGINE=InnoDB;""",
    
    """CREATE TABLE repayments (
        repayment_id     VARCHAR(20) PRIMARY KEY,
        loan_id          VARCHAR(20) NOT NULL,
        due_date         DATE NOT NULL,
        paid_date        DATE NULL,
        amount_due       DECIMAL(10,2) NOT NULL,
        amount_paid      DECIMAL(10,2) NOT NULL,
        days_late        INT DEFAULT 0,
        
        FOREIGN KEY (loan_id) REFERENCES loans(loan_id) ON DELETE RESTRICT,
        INDEX idx_due_date (due_date),
        INDEX idx_paid_date (paid_date),
        INDEX idx_days_late (days_late)
    ) ENGINE=InnoDB;"""
]

with engine.connect() as conn:
    conn.execute(text(f"USE {DB_NAME};"))
    
    for statement in schema_statements:
        conn.execute(text(statement))
        print(f"Executed: {statement.split('(')[0].strip() if '(' in statement else statement[:50]}...")

print("\n Professional Star-Schema Created Successfully with Proper Indexes!")

Executed: DROP TABLE IF EXISTS repayments;...
Executed: DROP TABLE IF EXISTS loans;...
Executed: DROP TABLE IF EXISTS customers;...
Executed: CREATE TABLE customers...
Executed: CREATE TABLE loans...
Executed: CREATE TABLE repayments...

 Professional Star-Schema Created Successfully with Proper Indexes!


#### Load Data from CSV

In [8]:
base_path = '../data/raw'

customers = pd.read_csv(f'{base_path}/customers.csv')
loans_df = pd.read_csv(f'{base_path}/loans.csv')
repayments_df = pd.read_csv(f'{base_path}/repayments.csv')

print(f"Loaded {len(customers):,} customers, {len(loans_df):,} loans, and {len(repayments_df):,} repayments")

Loaded 15,000 customers, 19,692 loans, and 115,072 repayments


#### Upload Data to MySQL

In [9]:
customers.to_sql('customers', engine, if_exists='append', index=False, chunksize=1000, method='multi')
loans_df.to_sql('loans', engine, if_exists='append', index=False, chunksize=1000, method='multi')
repayments_df.to_sql('repayments', engine, if_exists='append', index=False, chunksize=2000, method='multi')

print("All data successfully loaded into MySQL database!")

All data successfully loaded into MySQL database!


#### Final Validation

In [10]:
validation_sql = """
SELECT 
    (SELECT COUNT(*) FROM customers) as total_customers,
    (SELECT COUNT(*) FROM loans) as total_loans,
    (SELECT COUNT(*) FROM repayments) as total_repayments,
    ROUND((SELECT SUM(principal) FROM loans), 0) as total_disbursed_kes;
"""

with engine.connect() as conn:
    conn.execute(text(f"USE {DB_NAME};"))
    result = conn.execute(text(validation_sql)).fetchone()

print("=== FINAL DATABASE VALIDATION ===")
print(f"Total Customers   : {result[0]:,}")
print(f"Total Loans       : {result[1]:,}")
print(f"Total Repayments  : {result[2]:,}")
print(f"Total Disbursed   : KES {result[3]:,}")

=== FINAL DATABASE VALIDATION ===
Total Customers   : 15,000
Total Loans       : 19,692
Total Repayments  : 115,072
Total Disbursed   : KES 115,201,900
